# Advanced Python Set Update Operations

## Problems, Complete Solutions, Tests, and Best Practices

This notebook develops practical mastery of **in-place set mutation**:

- `set.update(*iterables)` / `|=`
- `set.intersection_update(*iterables)` / `&=`
- `set.difference_update(*iterables)` / `-=`
- `set.symmetric_difference_update(iterable)` / `^=`

The emphasis is not only on syntax, but also on object identity, aliasing, API design, correctness, edge cases, and production-style problem solving.

## Learning objectives

After completing the notebook, you should be able to:

1. Distinguish **mutation** from **rebinding**.
2. Preserve a set's identity when other parts of a program share the same object.
3. Select the correct update operation for union, intersection, difference, and symmetric difference.
4. Explain why method forms accept general iterables while operator forms normally require set-like operands.
5. Avoid common mistakes involving strings, unhashable values, non-associative difference, and multi-set symmetric difference.
6. Design functions that mutate caller-owned sets safely and predictably.
7. Use staging copies when an update must be validated before committing.

## 1. Quick reference

| Goal | Mutating method | Augmented operator | Meaning after mutation |
|---|---|---|---|
| Add every distinct item | `s.update(a, b, ...)` | `s |= other_set` | `s` becomes the union |
| Keep only common items | `s.intersection_update(a, b, ...)` | `s &= other_set` | `s` becomes the intersection |
| Remove specified items | `s.difference_update(a, b, ...)` | `s -= other_set` | `s` loses all items found in any operand |
| Toggle membership | `s.symmetric_difference_update(a)` | `s ^= other_set` | Items in exactly one of the two sets remain |

### Important method/operator distinction

The **method forms accept arbitrary iterables** of hashable elements. The binary operator forms are stricter and are intended for set-like operands.

All four mutating methods return `None`. That convention signals: **the object was changed in place**.

In [1]:
def show_identity(label, value):
    """Print a readable identity snapshot for demonstrations."""
    print(f"{label:<18} id={id(value)} value={value}")

s = {1, 2, 3}
alias = s

show_identity("before: s", s)
show_identity("before: alias", alias)

s.update([3, 4, 5])

show_identity("after: s", s)
show_identity("after: alias", alias)

assert s is alias
assert s == {1, 2, 3, 4, 5}

before: s          id=2245757287808 value={1, 2, 3}
before: alias      id=2245757287808 value={1, 2, 3}
after: s           id=2245757287808 value={1, 2, 3, 4, 5}
after: alias       id=2245757287808 value={1, 2, 3, 4, 5}


## 2. Mutation versus rebinding

These expressions can produce the same values but different object behavior:

```python
s = s | other       # computes a new set, then rebinds s
s |= other          # normally mutates the existing mutable set
s.update(other)     # explicitly mutates the existing set
```

When aliases exist, rebinding updates only one variable. Mutation is visible through every alias to the object.

In [2]:
original = {1, 2, 3}
alias = original
other = {3, 4}

old_id = id(original)
original = original | other

assert id(original) != old_id
assert original == {1, 2, 3, 4}
assert alias == {1, 2, 3}  # alias still points to the old object

mutated = {1, 2, 3}
mutated_alias = mutated
old_id = id(mutated)
mutated |= other

assert id(mutated) == old_id
assert mutated_alias == {1, 2, 3, 4}

## 3. Core examples and edge cases

### Example A — Methods can consume multiple iterable types

A method can update from lists, tuples, generators, dictionary views, and strings, provided the yielded elements are hashable.

In [3]:
s = {1, 2}
generator = (n for n in [2, 3, 4])
lookup = {4: "four", 5: "five"}

s.update([2, 3], (3, 4), generator, lookup.keys())
assert s == {1, 2, 3, 4, 5}
s

{1, 2, 3, 4, 5}

### Example B — `add` versus `update` with a string

A string is one hashable object, but it is also an iterable of characters.

In [4]:
whole_word = set()
whole_word.add("python")
assert whole_word == {"python"}

characters = set()
characters.update("python")
assert characters == set("python")

print("add:", whole_word)
print("update:", characters)

add: {'python'}
update: {'h', 'n', 'o', 'p', 'y', 't'}


### Example C — Variadic update methods

`difference_update(a, b, c)` removes the union of all supplied iterables. It does **not** reproduce arbitrary parenthesized difference expressions.

In [5]:
base = {1, 2, 3, 4, 5}
base.difference_update({2, 3}, [4], (99,))
assert base == {1, 5}

### Example D — Self-update behavior

Self-updates are legal and sometimes reveal the algebra clearly.

In [6]:
s = {1, 2, 3}
s |= s
assert s == {1, 2, 3}

s = {1, 2, 3}
s &= s
assert s == {1, 2, 3}

s = {1, 2, 3}
s -= s
assert s == set()

s = {1, 2, 3}
s ^= s
assert s == set()

## 4. Best practices

1. **Use a mutating method when identity matters.** This is especially important for shared state, function arguments, caches, registries, and observer-visible containers.
2. **Use the method form for general iterables.** It avoids unnecessary `set(...)` conversions.
3. **Never assign the result of an update method.** `s = s.update(items)` makes `s` become `None`.
4. **Treat strings deliberately.** Use `add(text)` for one string item and `update(text)` only when character expansion is intended.
5. **Validate before committing complex changes.** Apply proposed operations to a copy, validate, then commit with `clear()` and `update()` to preserve identity.
6. **Document mutation in function names or docstrings.** Callers should know whether a function changes an argument.
7. **Prefer assertions and invariant checks in exercises and tests.** Set order is not semantically meaningful, so test equality rather than printed order.
8. **Remember that elements must be hashable.** Convert mutable structures to immutable canonical forms such as tuples or `frozenset` when appropriate.

# Advanced Problems with Solutions

Each problem includes a specification, a complete solution, and executable checks.

## Problem 1 — Preserve aliases while merging capabilities

A permissions set is shared by multiple components. Implement `merge_capabilities_in_place` so that it:

- accepts any number of iterables,
- mutates the original set,
- preserves its object identity,
- returns `None`.

In [7]:
def merge_capabilities_in_place(target, *capability_sources):
    """Mutate target by adding capabilities from every iterable source."""
    target.update(*capability_sources)
    # Deliberately no return statement: mutating methods conventionally return None.

permissions = {"read"}
observer = permissions
before = id(permissions)

result = merge_capabilities_in_place(
    permissions,
    ["write", "read"],
    ("share",),
    (name for name in ["archive", "write"]),
)

assert result is None
assert id(permissions) == before
assert permissions is observer
assert permissions == {"read", "write", "share", "archive"}

## Problem 2 — Normalize and aggregate tags without splitting words

Records contain tag fields that may be:

- a single string,
- a list or tuple of strings,
- `None`.

Write an in-place aggregator that adds whole tags rather than accidentally adding individual characters.

In [8]:
def update_tags(target, raw_tags):
    """Normalize one tag field and mutate target with complete tag strings."""
    if raw_tags is None:
        return
    if isinstance(raw_tags, str):
        target.add(raw_tags)
        return
    target.update(raw_tags)

records = [
    {"id": 1, "tags": "python"},
    {"id": 2, "tags": ["sets", "python"]},
    {"id": 3, "tags": ("mutation", "best-practices")},
    {"id": 4, "tags": None},
]

all_tags = set()
for record in records:
    update_tags(all_tags, record["tags"])

assert all_tags == {"python", "sets", "mutation", "best-practices"}
assert "p" not in all_tags
all_tags

{'best-practices', 'mutation', 'python', 'sets'}

## Problem 3 — Reconcile effective permissions

A user's effective permissions must satisfy all of these rules:

1. Start from currently granted permissions.
2. Keep only permissions supported by the active plan.
3. Remove revoked, expired, and administrator-blocked permissions.

Mutate the original `granted` set so aliases observe the final result.

In [9]:
def reconcile_permissions_in_place(
    granted,
    plan_supported,
    revoked=(),
    expired=(),
    admin_blocked=(),
):
    """Mutate granted into the final effective permission set."""
    granted.intersection_update(plan_supported)
    granted.difference_update(revoked, expired, admin_blocked)


granted = {"read", "write", "share", "export", "admin"}
ui_reference = granted

reconcile_permissions_in_place(
    granted,
    plan_supported=["read", "write", "share", "export"],
    revoked={"share"},
    expired=["export"],
    admin_blocked=("admin",),
)

assert granted is ui_reference
assert granted == {"read", "write"}

## Problem 4 — Streaming deduplication with a live blocklist

Pages of identifiers arrive from a data source. A blocklist may grow while pages are processed.

Maintain one shared `accepted` set such that after every page:

- newly seen identifiers are added,
- all currently blocked identifiers are absent.

In [10]:
def ingest_page_in_place(accepted, page, blocked):
    """Add a page, then enforce the current blocklist."""
    accepted.update(page)
    accepted.difference_update(blocked)

pages = [
    [101, 102, 103, 101],
    [103, 104, 105],
    [105, 106, 107],
]

accepted = set()
shared_view = accepted
blocked = {102}

for index, page in enumerate(pages):
    if index == 2:
        blocked.update({104, 106})
    ingest_page_in_place(accepted, page, blocked)

assert accepted is shared_view
assert accepted == {101, 103, 105, 107}

## Problem 5 — Toggle feature flags

A toggle request means:

- enable a flag if it is currently disabled,
- disable it if it is currently enabled.

Implement this operation in one set-update statement.

In [11]:
def toggle_flags_in_place(enabled, requested_toggles):
    """Toggle each requested flag exactly once."""
    enabled.symmetric_difference_update(requested_toggles)

enabled = {"search", "billing", "dark-mode"}
toggle_flags_in_place(enabled, ["billing", "analytics", "dark-mode"])

assert enabled == {"search", "analytics"}

## Problem 6 — Symmetric difference across three sources: parity trap

For two sets, symmetric difference means “in exactly one set.” For three or more repeated symmetric differences, it means **odd membership count**, not necessarily membership in exactly one source.

Given three source sets:

1. Compute items appearing in an odd number of sources.
2. Compute items appearing in exactly one source.
3. Demonstrate that the two results can differ.

In [12]:
from collections import Counter

source_a = {"a", "b", "c", "x"}
source_b = {"b", "c", "d", "x"}
source_c = {"c", "d", "e", "x"}

# Odd parity: repeated symmetric difference.
odd_membership = set(source_a)
odd_membership.symmetric_difference_update(source_b)
odd_membership.symmetric_difference_update(source_c)

# Exactly one source: count memberships explicitly.
counts = Counter()
for source in (source_a, source_b, source_c):
    counts.update(source)
exactly_one = {item for item, count in counts.items() if count == 1}

assert odd_membership == {"a", "c", "e", "x"}
assert exactly_one == {"a", "e"}
assert odd_membership != exactly_one

print("odd membership:", odd_membership)
print("exactly one:", exactly_one)

odd membership: {'c', 'x', 'a', 'e'}
exactly one: {'a', 'e'}


## Problem 7 — Difference is not associative

Let:

```python
A = {1, 2, 3, 4, 5}
B = {2, 3, 6}
C = {3, 4}
```

Compute and compare:

- `A - (B - C)`
- `(A - B) - C`
- the result of `working.difference_update(B, C)`

Explain the relationship.

In [13]:
A = {1, 2, 3, 4, 5}
B = {2, 3, 6}
C = {3, 4}

first = A - (B - C)
second = (A - B) - C

working = set(A)
working.difference_update(B, C)

assert first == {1, 3, 4, 5}
assert second == {1, 5}
assert working == second

# Variadic difference_update removes every item found in B OR C:
assert working == A - (B | C)

**Conclusion:** `difference_update(B, C)` is equivalent to removing the union `B | C`, so it matches `(A - B) - C`. It does not match `A - (B - C)`.

## Problem 8 — Breadth-first search frontier management

Implement one layer expansion of a graph search. Mutate `visited` and return the next frontier.

Requirements:

- neighbors of every node in the current frontier are candidates,
- previously visited nodes must not re-enter the frontier,
- all newly discovered nodes must be added to `visited` in place.

In [14]:
def expand_frontier(graph, frontier, visited):
    """Return the next BFS frontier and mutate visited in place."""
    candidates = set()
    for node in frontier:
        candidates.update(graph.get(node, ()))

    candidates.difference_update(visited)
    visited.update(candidates)
    return candidates


graph = {
    "A": {"B", "C"},
    "B": {"A", "D", "E"},
    "C": {"A", "F"},
    "D": {"B"},
    "E": {"B", "F"},
    "F": {"C", "E"},
}

visited = {"A"}
reference = visited
frontier = {"A"}

frontier = expand_frontier(graph, frontier, visited)
assert frontier == {"B", "C"}
assert visited == {"A", "B", "C"}

frontier = expand_frontier(graph, frontier, visited)
assert frontier == {"D", "E", "F"}
assert visited is reference
assert visited == {"A", "B", "C", "D", "E", "F"}

## Problem 9 — Incremental search-index refresh

An index tracks document IDs. During a refresh:

- `discovered` IDs must be added,
- `deleted` IDs must be removed,
- only IDs in `readable_now` may remain.

The order matters. Implement a safe in-place refresh and test it.

In [15]:
def refresh_index_in_place(indexed, discovered, deleted, readable_now):
    """Mutate indexed to the final set of searchable documents."""
    indexed.update(discovered)
    indexed.difference_update(deleted)
    indexed.intersection_update(readable_now)

indexed = {1, 2, 3, 4}
monitor = indexed

refresh_index_in_place(
    indexed,
    discovered=[4, 5, 6, 7],
    deleted={2, 6},
    readable_now={1, 3, 4, 5, 8},
)

assert indexed is monitor
assert indexed == {1, 3, 4, 5}

## Problem 10 — Transactional update with validation

A live routing table is shared across a program. A proposed batch must satisfy:

- every route is a non-empty string,
- no forbidden route is present,
- at least one route remains after the change.

Do not partially mutate the live set if validation fails. Preserve identity when committing a valid change.

In [16]:
def apply_route_change_in_place(live_routes, *, add=(), remove=(), forbidden=()):
    """Validate on a staging copy, then commit while preserving identity."""
    candidate = live_routes.copy()
    candidate.update(add)
    candidate.difference_update(remove)

    if not candidate:
        raise ValueError("at least one route must remain")
    if not all(isinstance(route, str) and route for route in candidate):
        raise TypeError("every route must be a non-empty string")
    if candidate.intersection(forbidden):
        raise ValueError("candidate contains a forbidden route")

    # Commit in place so aliases keep observing the same object.
    live_routes.clear()
    live_routes.update(candidate)


routes = {"/", "/health"}
observer = routes
before = id(routes)

apply_route_change_in_place(
    routes,
    add=["/api", "/docs"],
    remove=["/health"],
    forbidden={"/admin"},
)

assert id(routes) == before
assert routes is observer
assert routes == {"/", "/api", "/docs"}

snapshot = routes.copy()
try:
    apply_route_change_in_place(
        routes,
        add=["/admin"],
        forbidden={"/admin"},
    )
except ValueError:
    pass
else:
    raise AssertionError("expected validation failure")

assert routes == snapshot  # no partial mutation occurred

## Problem 11 — Canonicalize unhashable records

A set cannot contain dictionaries because dictionaries are mutable and unhashable. Convert each small record dictionary into a canonical, hashable representation, then accumulate unique records in place.

Use `frozenset(record.items())` so dictionary item order does not affect equality.

In [17]:
def canonical_record(record):
    """Convert a flat dictionary with hashable values into a hashable value."""
    return frozenset(record.items())


def update_unique_records(target, records):
    target.update(canonical_record(record) for record in records)

records = [
    {"city": "Paris", "country": "FR"},
    {"country": "FR", "city": "Paris"},  # same logical record, different order
    {"city": "Sofia", "country": "BG"},
]

unique_records = set()
update_unique_records(unique_records, records)

assert len(unique_records) == 2
assert canonical_record({"city": "Paris", "country": "FR"}) in unique_records

## Problem 12 — Detect a rebinding bug

The function below is intended to mutate the caller's set, but it is incorrect:

```python
def broken_add(target, incoming):
    target = target | set(incoming)
```

Repair it, then prove with an alias that the repair preserves object identity.

In [18]:
def broken_add(target, incoming):
    target = target | set(incoming)  # local rebinding only


def fixed_add(target, incoming):
    target.update(incoming)

items = {1, 2}
alias = items
broken_add(items, [2, 3])
assert items == {1, 2}
assert alias == {1, 2}

before = id(items)
fixed_add(items, [2, 3])
assert id(items) == before
assert items is alias
assert items == {1, 2, 3}

## Problem 13 — Compute Jaccard similarity without corrupting inputs

Jaccard similarity is:

\[
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
\]

Use mutating set operations internally for efficiency and clarity, but do not mutate the caller's input sets. Define the empty/empty similarity as `1.0`.

In [19]:
def jaccard_similarity(left, right):
    """Return Jaccard similarity without changing either input."""
    intersection = set(left)
    intersection.intersection_update(right)

    union = set(left)
    union.update(right)

    if not union:
        return 1.0
    return len(intersection) / len(union)

A = {1, 2, 3}
B = {2, 3, 4, 5}
A_before = A.copy()
B_before = B.copy()

score = jaccard_similarity(A, B)
assert score == 2 / 5
assert A == A_before
assert B == B_before
assert jaccard_similarity(set(), set()) == 1.0

## Problem 14 — Capstone: paged event ingestion pipeline

Build a reusable ingestion function for paged events. Each event is a dictionary with `id`, `kind`, and `active` fields.

Rules:

1. Only active events are candidates.
2. Only allowed kinds are accepted.
3. Blocked IDs are removed even if they appeared on an earlier page.
4. Duplicate IDs collapse naturally.
5. The caller-owned output set must be mutated in place.

Return a statistics dictionary while preserving the output set's identity.

In [20]:
def ingest_event_pages_in_place(
    accepted_ids,
    pages,
    *,
    allowed_kinds,
    blocked_ids=(),
):
    """Mutate accepted_ids and return ingestion statistics."""
    allowed_kinds = set(allowed_kinds)
    blocked_ids = set(blocked_ids)

    seen_rows = 0
    active_rows = 0
    eligible_rows = 0

    for page in pages:
        page_candidates = set()

        for event in page:
            seen_rows += 1
            if not event.get("active", False):
                continue
            active_rows += 1
            if event.get("kind") not in allowed_kinds:
                continue
            eligible_rows += 1
            page_candidates.add(event["id"])

        page_candidates.difference_update(blocked_ids)
        accepted_ids.update(page_candidates)

        # Re-apply the blocklist so previously accepted IDs are also removed.
        accepted_ids.difference_update(blocked_ids)

    return {
        "seen_rows": seen_rows,
        "active_rows": active_rows,
        "eligible_rows": eligible_rows,
        "unique_accepted": len(accepted_ids),
    }


pages = [
    [
        {"id": 1, "kind": "deploy", "active": True},
        {"id": 2, "kind": "audit", "active": True},
        {"id": 3, "kind": "debug", "active": True},
        {"id": 1, "kind": "deploy", "active": True},
    ],
    [
        {"id": 4, "kind": "deploy", "active": False},
        {"id": 5, "kind": "audit", "active": True},
        {"id": 6, "kind": "deploy", "active": True},
        {"id": 2, "kind": "audit", "active": True},
    ],
]

accepted_ids = {99}
observer = accepted_ids
stats = ingest_event_pages_in_place(
    accepted_ids,
    pages,
    allowed_kinds={"deploy", "audit"},
    blocked_ids={2, 99},
)

assert accepted_ids is observer
assert accepted_ids == {1, 5, 6}
assert stats == {
    "seen_rows": 8,
    "active_rows": 7,
    "eligible_rows": 6,
    "unique_accepted": 3,
}
stats

{'seen_rows': 8, 'active_rows': 7, 'eligible_rows': 6, 'unique_accepted': 3}

# Additional Challenge Problems

Try these before reading the compact solutions below.

## Challenge A — In-place whitelist refresh

Given a shared set `current`, mutate it so it contains all IDs from `incoming` that are also in `whitelist`, excluding `quarantined`.

In [21]:
def whitelist_refresh_in_place(current, incoming, whitelist, quarantined):
    current.clear()
    current.update(incoming)
    current.intersection_update(whitelist)
    current.difference_update(quarantined)

current = {999}
alias = current
whitelist_refresh_in_place(
    current,
    incoming=[1, 2, 2, 3, 4, 5],
    whitelist={2, 3, 4, 6},
    quarantined={3},
)
assert current is alias
assert current == {2, 4}

## Challenge B — Mutate a set to exactly the target contents

Write a small helper that replaces the contents of a set without replacing the set object itself.

In [22]:
def replace_contents_in_place(target, new_values):
    """Preserve target identity while replacing all of its contents."""
    target.clear()
    target.update(new_values)

s = {1, 2, 3}
alias = s
replace_contents_in_place(s, [10, 20, 20])
assert s is alias
assert s == {10, 20}

## Challenge C — Update from dictionary values, not keys

Remember that iterating over a dictionary yields keys. Add the dictionary's values to a set.

In [23]:
status_by_id = {101: "queued", 102: "running", 103: "queued"}
statuses = set()
statuses.update(status_by_id.values())
assert statuses == {"queued", "running"}

# Common mistakes checklist

### Mistake 1

```python
s = s.update(items)
```

`update` returns `None`, so this destroys the reference held by `s`.

### Mistake 2

```python
words.update("python")
```

This adds characters. Use `words.add("python")` for one complete word.

### Mistake 3

Assuming repeated symmetric difference across three or more sets means “exactly one source.” It means odd parity.

### Mistake 4

Assuming difference is associative. Parentheses change the result.

### Mistake 5

Using rebinding inside a function when the caller expects mutation.

### Mistake 6

Trying to add a list, set, or dictionary as an element. Set elements must be hashable.

# Final mastery test

The function below combines all four update families. Read the specification, predict the result, then run the cell.

Starting from `state`:

1. union in `discovered`,
2. keep only `supported`,
3. remove `blocked`,
4. toggle `experimental`.

In [24]:
def reconcile_state_in_place(
    state,
    *,
    discovered=(),
    supported=(),
    blocked=(),
    experimental=(),
):
    state.update(discovered)
    state.intersection_update(supported)
    state.difference_update(blocked)
    state.symmetric_difference_update(experimental)

state = {"a", "b", "legacy"}
alias = state

reconcile_state_in_place(
    state,
    discovered=["b", "c", "d"],
    supported={"a", "b", "c", "d", "e"},
    blocked={"b"},
    experimental={"c", "e"},
)

# After union:        {a, b, c, d, legacy}
# After intersection: {a, b, c, d}
# After difference:   {a, c, d}
# After toggle:       {a, d, e}
assert state is alias
assert state == {"a", "d", "e"}
state

{'a', 'd', 'e'}

# Summary

- Use `update` / `|=` to add members.
- Use `intersection_update` / `&=` to retain only common members.
- Use `difference_update` / `-=` to remove members.
- Use `symmetric_difference_update` / `^=` to toggle two-set membership.
- Prefer methods when consuming arbitrary iterables or multiple operands.
- Preserve identity when aliases or external observers matter.
- Validate on a copy before committing risky multi-step mutations.
- Test sets by equality, not by display order.